In [ ]:
library(ggplot2)

indir <- "/path/to/nature_study/ATAC_donor_level"
outdir <- "/path/to/nature_study/ATAC_donor_level"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

hsc_file <- file.path(indir, "HSC_MPP_disomyHigher_motifs_raw_mlog10Padj.tsv")
bprog_file <- file.path(indir, "B_primed_prog_disomyHigher_motifs_raw_mlog10Padj.tsv")

read_motif_table <- function(f) {
  x <- read.delim(f, check.names = FALSE, stringsAsFactors = FALSE)
  nms <- colnames(x)

  motif_col <- nms[grep("motif|name|tf", nms, ignore.case = TRUE)][1]
  score_col <- nms[grep("mlog10|neglog10|padj|fdr|score", nms, ignore.case = TRUE)][1]

  if (is.na(motif_col) || is.na(score_col)) {
    stop(paste("Could not detect motif or score column in", f,
               "\nColumns are:", paste(nms, collapse = ", ")))
  }

  df <- data.frame(
    motif = x[[motif_col]],
    score = as.numeric(x[[score_col]]),
    stringsAsFactors = FALSE
  )

  df <- df[!is.na(df$motif) & !is.na(df$score), , drop = FALSE]

  # optional cleanup of motif labels
  df$motif <- gsub("_\\d+$", "", df$motif)
  df$motif <- gsub("\\.\\d+$", "", df$motif)

  df
}

make_motif_plot <- function(df, title_text, fill_color, top_n = 8) {
  df <- df[order(df$score, decreasing = TRUE), , drop = FALSE]
  df <- head(df, top_n)
  df$motif <- factor(df$motif, levels = rev(df$motif))

  ggplot(df, aes(x = score, y = motif)) +
    geom_col(fill = fill_color, width = 0.7) +
    theme_classic() +
    theme(
      axis.title.y = element_blank(),
      axis.title.x = element_text(size = 15),
      axis.text.x = element_text(size = 13),
      axis.text.y = element_text(size = 13),
      plot.title = element_text(size = 13, face = "bold", hjust = 0.5)
    ) +
    labs(
      title = title_text,
      x = expression(-log[10]("adj. P"))
    )
}

hsc_df <- read_motif_table(hsc_file)
bprog_df <- read_motif_table(bprog_file)

p_hsc <- make_motif_plot(
  hsc_df,
  title_text = "HSC_MPP disomy-higher motifs",
  fill_color = "#4C78A8",
  top_n = 8
)

p_bprog <- make_motif_plot(
  bprog_df,
  title_text = "B_primed_prog disomy-higher motifs",
  fill_color = "#F8766D",
  top_n = 8
)

print(p_hsc)
print(p_bprog)

ggsave(
  filename = file.path(outdir, "HSC_MPP_disomyHigher_motifs_top8.pdf"),
  plot = p_hsc,
  width = 5.2,
  height = 3.8
)

ggsave(
  filename = file.path(outdir, "HSC_MPP_disomyHigher_motifs_top8.tiff"),
  plot = p_hsc,
  width = 5.2,
  height = 3.8,
  units = "in",
  dpi = 600,
  compression = "lzw"
)

ggsave(
  filename = file.path(outdir, "B_primed_prog_disomyHigher_motifs_top8.pdf"),
  plot = p_bprog,
  width = 5.2,
  height = 3.8
)

ggsave(
  filename = file.path(outdir, "B_primed_prog_disomyHigher_motifs_top8.tiff"),
  plot = p_bprog,
  width = 5.2,
  height = 3.8,
  units = "in",
  dpi = 600,
  compression = "lzw"
)

In [ ]:
hsc_keep <- c("BCL11A", "BCL11B", "IRF4", "IRF8", "IRF1", "SPI1", "SPIB", "EBF1")
bprog_keep <- c("TCF3", "PAX5", "TCF12", "EBF1", "TFAP4", "LYL1", "NHLH1", "NHLH2")

hsc_df2 <- hsc_df[hsc_df$motif %in% hsc_keep, , drop = FALSE]
bprog_df2 <- bprog_df[bprog_df$motif %in% bprog_keep, , drop = FALSE]

hsc_df2 <- hsc_df2[match(hsc_keep, hsc_df2$motif), , drop = FALSE]
bprog_df2 <- bprog_df2[match(bprog_keep, bprog_df2$motif), , drop = FALSE]

hsc_df2 <- hsc_df2[!is.na(hsc_df2$motif), , drop = FALSE]
bprog_df2 <- bprog_df2[!is.na(bprog_df2$motif), , drop = FALSE]

p_hsc <- make_motif_plot(hsc_df2, "HSC MPP motifs higher in D", "#4C78A8", top_n = nrow(hsc_df2))
p_bprog <- make_motif_plot(bprog_df2, "B progenitor motifs higher in D", "#F8766D", top_n = nrow(bprog_df2))

In [ ]:
print(p_hsc)
print(p_bprog)

ggsave(
  filename = file.path(outdir, "HSC_MPP_disomyHigher_motifs_selected.pdf"),
  plot = p_hsc,
  width = 3.5,
  height = 3.8
)

ggsave(
  filename = file.path(outdir, "HSC_MPP_disomyHigher_motifs_selected.tiff"),
  plot = p_hsc,
  width = 3.5,
  height = 3.8,
  units = "in",
  dpi = 600,
  compression = "lzw"
)

ggsave(
  filename = file.path(outdir, "B_primed_prog_disomyHigher_motifs_selected.pdf"),
  plot = p_bprog,
  width = 3.5,
  height = 3.8
)

ggsave(
  filename = file.path(outdir, "B_primed_prog_disomyHigher_motifs_selected.tiff"),
  plot = p_bprog,
  width = 3.5,
  height = 3.8,
  units = "in",
  dpi = 600,
  compression = "lzw"
)